## Importations

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split

# models
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


## Chargement du dataset Purchase-100

In [2]:
dataset = pd.read_csv("./sample_data/dataset_purchase", header=None)
dataset= dataset.values
X, y = dataset[:, 1:], dataset[:, 0]
y = y - 1
X = X[:20000]
y = y[:20000]

## Découpage des donnees

In [3]:


X_target, X_shadow, y_target, y_shadow = train_test_split(X,y, test_size= 0.5, random_state= 42)

X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
    X_target, y_target, test_size=0.5, random_state=42
)

X_train_shadow, X_out_shadow, y_train_shadow, y_out_shadow = train_test_split(
    X_shadow, y_shadow, test_size=0.5, random_state=42
)


## Construction & l'entrainement du target model

In [4]:

target_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=20,
    n_jobs=1
)

target_model.fit(X_train_target, y_train_target)


RandomForestClassifier(max_depth=20, n_estimators=1000, n_jobs=1)

## Overfiting du target model

In [5]:
train_acc = target_model.score(X_train_target, y_train_target)
test_acc = target_model.score(X_test_target, y_test_target)

print(train_acc, test_acc)
print("overfitting =", train_acc - test_acc)

1.0 0.3504
overfitting = 0.6496


## Construction & l'entraintement du sous shadow model (Random Forest )

In [6]:


shadow_submodel_rfc = RandomForestClassifier(
    n_estimators=1000,
    max_depth=20,
    n_jobs=1
)

shadow_submodel_rfc.fit(X_train_shadow, y_train_shadow)


RandomForestClassifier(max_depth=20, n_estimators=1000, n_jobs=1)

## Construction et l'entrainement du sous shadow model (Multilayer Perceptron)

In [7]:

shadow_submodel_mlp = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation='relu',
    solver='adam',
    max_iter=200,
    random_state=42
)

shadow_submodel_mlp.fit(X_train_shadow, y_train_shadow)

MLPClassifier(hidden_layer_sizes=(128,), random_state=42)

## Construction et l'entrainement du sous shadow model(Logistic Regression)

In [8]:

scaler = StandardScaler()
X_train_shadow = scaler.fit_transform(X_train_shadow)
X_out_shadow = scaler.transform(X_out_shadow)

shadow_submodel_lr = LogisticRegression(
    max_iter=1000,
    n_jobs=1,
    random_state=42
)

shadow_submodel_lr.fit(X_train_shadow, y_train_shadow)

LogisticRegression(max_iter=1000, n_jobs=1, random_state=42)

## Construction de dataset du modele d'attaque

In [9]:
shadow_models=[shadow_submodel_lr, shadow_submodel_mlp, shadow_submodel_rfc]
attack_features = []
attack_labels = []

for model in shadow_models:
    # on interroge le shadow model par son train_dataset
    feature_vector_in = model.predict_proba(X_train_shadow)
    top3_in = np.sort(feature_vector_in, axis=1)[:, -3:]
    attack_features.append(top3_in)
    attack_labels.append(np.ones(len(top3_in)))

    # on interroge le shadow model par son out_dataset
    feature_vector_out = model.predict_proba(X_out_shadow)
    top3_out = np.sort(feature_vector_out, axis=1)[:, -3:]
    attack_features.append(top3_out)
    attack_labels.append(np.zeros(len(top3_out)))

attack_features = np.vstack(attack_features)
attack_labels = np.hstack(attack_labels)

## construction & l'entrainement du modele d'attaque

In [10]:
attack_model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    solver='adam',
    batch_size=128,
    max_iter=200,
    random_state=42
)

attack_model.fit(attack_features, attack_labels)

MLPClassifier(batch_size=128, hidden_layer_sizes=(64,), random_state=42)

## Preparer les donnees pour realiser l'attaque

In [11]:
target_features =[]
target_labels = []

# on interroge le target model par son train_dataset
feature_vector_in = target_model.predict_proba(X_train_target)
top3_in = np.sort(feature_vector_in, axis=1)[:, -3:]

# on interroge le target model par son test_dataset
feature_vector_out = target_model.predict_proba(X_test_target)
top3_out = np.sort(feature_vector_out, axis=1)[:, -3:]

target_features = np.vstack((top3_in, top3_out))
target_labels = np.hstack((
    np.ones(len(top3_in)),
    np.zeros(len(top3_out))
))

## Affichage de la precison et recall de l'attaque

In [12]:
target_labels_predicted= attack_model.predict(target_features)

precision = precision_score(target_labels,target_labels_predicted )
recall = recall_score(target_labels,target_labels_predicted )

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.99
Recall: 0.98
